# AI Interview Coach - Evaluation & RLAIF Scoring

This notebook is designed to run the 35B judge model (`Qwen/Qwen3.6-35B-A3B`) using 4-bit quantization on Google Colab Pro (L4 or A100 GPU).

## 1. Mount Google Drive
Mounting Drive ensures that your results (`.jsonl` files) are saved permanently even if the Colab runtime is disconnected.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Create a project directory if it doesn't exist
import os
PROJECT_PATH = "/content/drive/MyDrive/ai-interview-coach"
if not os.path.exists(PROJECT_PATH):
    !git clone https://github.com/dcyforjob2020/ai-interview-coach.git "{PROJECT_PATH}"

%cd "{PROJECT_PATH}"
# Pull latest changes if it already existed
!git pull origin main

## 2. Setup Environment

**CRITICAL**: You must upgrade core libraries to avoid compatibility bugs with 4-bit quantization. 
After running the cell below, **restart your runtime** (Runtime -> Restart session) and then resume from Step 3.

In [ ]:
# 1. Install project dependencies
!pip install -r requirements.txt

# 2. Force upgrade core libraries to latest compatible versions
!pip install -U transformers accelerate bitsandbytes

print("\n" + "="*60)
print("SETUP COMPLETE. PLEASE RESTART THE RUNTIME NOW")
print("(Go to 'Runtime' -> 'Restart session' in the menu bar)")
print("="*60)

## 3. Baseline Feedback Scoring

This will score the initial feedback generated by the base model. Results will be saved to `baseline/baseline_scores.jsonl` in your Google Drive.

In [ ]:
!python eval/score_feedback.py \
    --input baseline/baseline_outputs.jsonl \
    --output baseline/baseline_scores.jsonl \
    --feedback-field baseline_feedback \
    --quantize

## 4. Preference Labeling (RLAIF)

Run this after generating candidates to create the preference dataset.

In [ ]:
!python eval/score_preferences.py \
    --input train/preference_candidates.jsonl \
    --output train/preference_pairs.jsonl \
    --quantize

## 5. Summary & Verification
Check that the results were written correctly.

In [ ]:
import os
if os.path.exists("baseline/baseline_scores.jsonl"):
    print("Success! Baseline scores saved to Drive.")
    !head -n 2 baseline/baseline_scores.jsonl